##### Copyright 2019 The TensorFlow Authors.

Licensed under the Apache License, Version 2.0 (the "License");

In [10]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Image segmentation

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/tutorials/images/segmentation">
    <img src="https://www.tensorflow.org/images/tf_logo_32px.png" />
    View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/images/segmentation.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/docs/blob/master/site/en/tutorials/images/segmentation.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
    View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/docs/site/en/tutorials/images/segmentation.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

This tutorial focuses on the task of image segmentation, using a modified <a href="https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/" class="external">U-Net</a>.

## What is image segmentation?

In an image classification task, the network assigns a label (or class) to each input image. However, suppose you want to know the shape of that object, which pixel belongs to which object, etc. In this case, you need to assign a class to each pixel of the image—this task is known as segmentation. A segmentation model returns much more detailed information about the image. Image segmentation has many applications in medical imaging, self-driving cars and satellite imaging, just to name a few.

This tutorial uses the [Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/) ([Parkhi et al, 2012](https://www.robots.ox.ac.uk/~vgg/publications/2012/parkhi12a/parkhi12a.pdf)). The dataset consists of images of 37 pet breeds, with 200 images per breed (~100 each in the training and test splits). Each image includes the corresponding labels, and pixel-wise masks. The masks are class-labels for each pixel. Each pixel is given one of three categories:

- Class 1: Pixel belonging to the pet.
- Class 2: Pixel bordering the pet.
- Class 3: None of the above/a surrounding pixel.

In [11]:
# !pip install git+https://github.com/tensorflow/examples.git
# !pip install -U keras
# !pip install -q tensorflow_datasets
# !pip install -q -U tensorflow-text tensorflow

^C
  Cloning https://github.com/tensorflow/examples.git to c:\users\osami\appdata\local\temp\pip-req-build-gxtea9jr
  Resolved https://github.com/tensorflow/examples.git to commit 79e40789448a066b645598c8642464d170abfc37
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/tensorflow/examples.git 'C:\Users\osami\AppData\Local\Temp\pip-req-build-gxtea9jr'


^C


ERROR: Could not find a version that satisfies the requirement tensorflow-text (from versions: none)
ERROR: No matching distribution found for tensorflow-text


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 26.6 MB/s  0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.11.2
    Uninstalling keras-3.11.2:
      Successfully uninstalled keras-3.11.2


In [12]:
import numpy as np

import tensorflow as tf
import tensorflow_datasets as tfds

In [13]:
import sys
print(sys.executable)  # Full path to Python executable
print(sys.version)     # Python version

d:\Anaconda3\envs\resistor-2\python.exe
3.11.14 | packaged by conda-forge | (main, Oct 22 2025, 22:35:28) [MSC v.1944 64 bit (AMD64)]


In [14]:
from tensorflow_examples.models.pix2pix import pix2pix

from IPython.display import clear_output
import matplotlib.pyplot as plt

## Download the Oxford-IIIT Pets dataset

The dataset is [available from TensorFlow Datasets](https://www.tensorflow.org/datasets/catalog/oxford_iiit_pet). The segmentation masks are included in version 3+.

In [15]:
train_path = "/content/data/training.tfrecord-0-1"
val_path   = "/content/data/validation.tfrecord-0-1"


In [17]:
import os

print(os.listdir("/content"))
print("--------")
print(os.listdir("/content/data") if os.path.exists("/content/data") else "data folder DOES NOT exist")


['data']
--------
[]


In [18]:
import os
os.makedirs("/content/data", exist_ok=True)
print("Created /content/data")


Created /content/data


In [19]:
print(os.listdir("/content/data"))



[]


In [20]:
# Your real label map based on label_map.pbtxt
ID_TO_NAME = {
    0: "Background",
    1: "Gold",
    2: "Orange",
    3: "Green",
    4: "Brown",
    # 5 was duplicate background → ignore
    6: "Blue",
    7: "Yellow",
    8: "Black",
    9: "White",
    10: "Grey",
    11: "Red",
    12: "Violet"
}


In [21]:
RESISTOR_COLORS_RGB = {
    "Background": (0, 0, 0),

    "Black":  (255, 105, 180),
    "Brown":  (101, 67, 33),
    "Red":    (255, 0, 0),
    "Orange": (255, 165, 0),
    "Yellow": (255, 255, 0),
    "Green":  (0, 128, 0),
    "Blue":   (0, 0, 255),
    "Violet": (148, 0, 211),
    "Grey":   (128, 128, 128),
    "White":  (255, 255, 255),
    "Gold":   (212, 175, 55),
    "Silver": (192, 192, 192)
}


In [22]:
feature_description = {
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/object/mask": tf.io.VarLenFeature(tf.string),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

def parse_example(example_proto):
    ex = tf.io.parse_single_example(example_proto, feature_description)

    # Decode image
    img = tf.image.decode_png(ex["image/encoded"], channels=3)
    img = tf.cast(img, tf.float32)

    # --- Masks and Class IDs ---
    mask_bytes = tf.sparse.to_dense(ex["image/object/mask"], default_value=b"")
    labels = tf.sparse.to_dense(ex["image/object/class/label"], default_value=0)

    # If no masks → blank mask
    def empty_mask():
        h = tf.shape(img)[0]
        w = tf.shape(img)[1]
        return tf.zeros((h, w, 1), tf.int32)

    # Build semantic mask (pure TF operations)
    def build_mask():
        # Decode mask images
        masks = tf.map_fn(
            lambda b: tf.cast(tf.image.decode_png(b, channels=1) > 0, tf.int32),
            mask_bytes,
            fn_output_signature=tf.TensorSpec(shape=(None, None, 1), dtype=tf.int32)
        )  # shape: (N, H, W, 1)

        # Broadcast class IDs → int32
        class_ids = tf.cast(
            tf.reshape(labels, (-1, 1, 1, 1)),
            tf.int32
        )

        # Multiply masks × class id
        masks = masks * class_ids  # (N, H, W, 1)

        # Reduce max → semantic mask
        sem = tf.reduce_max(masks, axis=0)  # (H, W, 1)
        return sem

    mask = tf.cond(tf.size(mask_bytes) > 0, build_mask, empty_mask)

    return img, mask




In [23]:
IMG_SIZE = (256, 256)   # keep this consistent with your model input

def preprocess(img, mask):
    # ---- Image preprocessing ----
    # img: (H, W, 3), uint8 0–255  → float32 0–1, resized
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0

    # ---- Mask preprocessing ----
    # mask: (H, W, 1), int32 class IDs
    # resize with NEAREST so labels don’t get mixed
    mask = tf.image.resize(mask, IMG_SIZE, method="nearest")
    mask = tf.cast(mask, tf.int32)

    # merge duplicate background label (5) into 0
    mask = tf.where(mask == 5, 0, mask)

    return img, mask





In [24]:
train_ds = (tf.data.TFRecordDataset(train_path)
            .map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
            .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(200)
            .batch(4)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.TFRecordDataset(val_path)
          .map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
          .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(4)
          .prefetch(tf.data.AUTOTUNE))




In [25]:
import os
print(os.listdir("/content"))
print("-------")
print(os.listdir("/content/data") if os.path.exists("/content/data") else "Data folder NOT found")


['data']
-------
[]


In [26]:
for img, mask in train_ds.take(1):
    print("dtype:", mask.dtype)
    print("unique:", np.unique(mask.numpy()))


NotFoundError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} NewRandomAccessFile failed to Create/Open: /content/data/training.tfrecord-0-1 : The system cannot find the file specified.
; No such file or directory [Op:IteratorGetNext] name: 

 In addition, the image color values are normalized to the `[0, 1]` range. Finally, as mentioned above the pixels in the segmentation mask are labeled either {1, 2, 3}. For the sake of convenience, subtract 1 from the segmentation mask, resulting in labels that are : {0, 1, 2}.

In [27]:
IMG_SIZE = (256, 256)   # use high-res now that we’re on GPU

def normalize(img, mask):
    # Resize
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0

    mask = tf.image.resize(mask, IMG_SIZE, method="nearest")
    mask = tf.cast(mask, tf.int32)

    return img, mask

BATCH_SIZE = 4  # GPU can handle this; if OOM, drop to 2

train_ds = (tf.data.TFRecordDataset(train_path)
            .map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
            .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(300)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.TFRecordDataset(val_path)
          .map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
          .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))


In [28]:
for imgs, masks in train_ds.take(1):
    print("Images:", imgs.shape, "Masks:", masks.shape)
    print("Unique mask IDs:", np.unique(masks[0].numpy()))


NotFoundError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} NewRandomAccessFile failed to Create/Open: /content/data/training.tfrecord-0-1 : The system cannot find the file specified.
; No such file or directory [Op:IteratorGetNext] name: 

In [29]:
def normalize(input_image, input_mask):
  input_image = tf.cast(input_image, tf.float32) / 255.0
  input_mask -= 1
  return input_image, input_mask

In [30]:
def load_image(datapoint):
  input_image = tf.image.resize(datapoint['image'], (128, 128))
  input_mask = tf.image.resize(
    datapoint['segmentation_mask'],
    (128, 128),
    method = tf.image.ResizeMethod.NEAREST_NEIGHBOR,
  )

  input_image, input_mask = normalize(input_image, input_mask)

  return input_image, input_mask

The dataset already contains the required training and test splits, so continue to use the same splits:

The following class performs a simple augmentation by randomly-flipping an image.
Go to the [Image augmentation](data_augmentation.ipynb) tutorial to learn more.


In [31]:
class Augment(tf.keras.layers.Layer):
  def __init__(self, seed=42):
    super().__init__()
    # both use the same seed, so they'll make the same random changes.
    self.augment_inputs = tf.keras.layers.RandomFlip(mode="horizontal", seed=seed)
    self.augment_labels = tf.keras.layers.RandomFlip(mode="horizontal", seed=seed)

  def call(self, inputs, labels):
    inputs = self.augment_inputs(inputs)
    labels = self.augment_labels(labels)
    return inputs, labels

Build the input pipeline, applying the augmentation after batching the inputs:

Visualize an image example and its corresponding mask from the dataset:

In [32]:
def display(display_list):
  plt.figure(figsize=(15, 15))

  title = ['Input Image', 'True Mask', 'Predicted Mask']

  for i in range(len(display_list)):
    plt.subplot(1, len(display_list), i+1)
    plt.title(title[i])
    plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
    plt.axis('off')
  plt.show()

In [33]:
import os

print("Content:", os.listdir("/content"))
print("Data folder:", os.listdir("/content/data") if os.path.exists("/content/data") else "NO DATA FOLDER")


Content: ['data']
Data folder: []


In [34]:
import os, shutil

os.makedirs("/content/data", exist_ok=True)

shutil.move("/content/training.tfrecord-0-1", "/content/data/")
shutil.move("/content/validation.tfrecord-0-1", "/content/data/")
shutil.move("/content/label_map.pbtxt", "/content/data/")
shutil.move("/content/color_map.json", "/content/data/")


FileNotFoundError: [Errno 2] No such file or directory: '/content/training.tfrecord-0-1'

## Define the model
The model being used here is a modified [U-Net](https://arxiv.org/abs/1505.04597). A U-Net consists of an encoder (downsampler) and decoder (upsampler). To learn robust features and reduce the number of trainable parameters, use a pretrained model—[MobileNetV2](https://arxiv.org/abs/1801.04381)—as the encoder. For the decoder, you will use the upsample block, which is already implemented in the [pix2pix](https://github.com/tensorflow/examples/blob/master/tensorflow_examples/models/pix2pix/pix2pix.py) example in the TensorFlow Examples repo. (Check out the [pix2pix: Image-to-image translation with a conditional GAN](../generative/pix2pix.ipynb) tutorial in a notebook.)


As mentioned, the encoder is a pretrained MobileNetV2 model. You will use the model from `tf.keras.applications`. The encoder consists of specific outputs from intermediate layers in the model. Note that the encoder will not be trained during the training process.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=[128, 128, 3], include_top=False)

# Use the activations of these layers
layer_names = [
    'block_1_expand_relu',   # 64x64
    'block_3_expand_relu',   # 32x32
    'block_6_expand_relu',   # 16x16
    'block_13_expand_relu',  # 8x8
    'block_16_project',      # 4x4
]
base_model_outputs = [base_model.get_layer(name).output for name in layer_names]

# Create the feature extraction model
down_stack = tf.keras.Model(inputs=base_model.input, outputs=base_model_outputs)

down_stack.trainable = False

The decoder/upsampler is simply a series of upsample blocks implemented in TensorFlow examples:

In [ ]:
up_stack = [
    pix2pix.upsample(512, 3),  # 4x4 -> 8x8
    pix2pix.upsample(256, 3),  # 8x8 -> 16x16
    pix2pix.upsample(128, 3),  # 16x16 -> 32x32
    pix2pix.upsample(64, 3),   # 32x32 -> 64x64
]

In [ ]:
def unet_model(output_channels:int):
  inputs = tf.keras.layers.Input(shape=[128, 128, 3])

  # Downsampling through the model
  skips = down_stack(inputs)
  x = skips[-1]
  skips = reversed(skips[:-1])

  # Upsampling and establishing the skip connections
  for up, skip in zip(up_stack, skips):
    x = up(x)
    concat = tf.keras.layers.Concatenate()
    x = concat([x, skip])

  # Last layer: NOW with softmax so y_pred are probabilities
  last = tf.keras.layers.Conv2DTranspose(
      filters=output_channels,
      kernel_size=3,
      strides=2,
      padding="same",
      activation="softmax"    # 👈 IMPORTANT
  )

  x = last(x)

  return tf.keras.Model(inputs=inputs, outputs=x)


In [ ]:
# new class weights cell


In [ ]:
# Your dataset has 13 classes: 0..12
OUTPUT_CLASSES = 13

# Class weights from your earlier counts (you can tweak later)
class_weights = tf.constant([
    0.29,  # 0 Background
    0.85,  # 1 Gold
    1.15,  # 2 Orange
    2.30,  # 3 Green
    0.90,  # 4 Brown
    0.30,  # 5 Body / background2
    1.40,  # 6 Blue
    2.80,  # 7 Yellow
    0.90,  # 8 Black
    2.20,  # 9 White
    2.40,  # 10 Grey
    0.90,  # 11 Red
    1.90,  # 12 Violet
], dtype=tf.float32)

def weighted_ce(y_true, y_pred):
    # y_true: (B,H,W,1)
    y_true = tf.squeeze(y_true, axis=-1)  # (B,H,W)
    y_true = tf.cast(y_true, tf.int32)

    ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)

    pix_weights = tf.gather(class_weights, y_true)
    pix_weights = tf.cast(pix_weights, tf.float32)
    return tf.reduce_mean(pix_weights * ce)

def dice_loss(y_true, y_pred, smooth=1):
    # y_true: (B,H,W,1)
    y_true = tf.squeeze(y_true, axis=-1)
    y_true = tf.cast(y_true, tf.int32)     # ⭐ REQUIRED FIX
    y_true = tf.one_hot(y_true, OUTPUT_CLASSES)


    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2])
    union = tf.reduce_sum(y_true + y_pred, axis=[1,2])

    dice = (2 * intersection + smooth) / (union + smooth)
    return 1 - tf.reduce_mean(dice)

def total_loss(y_true, y_pred):
    return weighted_ce(y_true, y_pred) + 0.5 * dice_loss(y_true, y_pred)


In [ ]:
from tensorflow.keras import layers, Model

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return x

def mid_unet(input_shape=(256,256,3), output_channels=13):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = conv_block(inputs, 32); p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, 64);     p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, 128);    p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, 256);    p4 = layers.MaxPooling2D()(c4)

    # Bottleneck
    bn = conv_block(p4, 512)

    # Decoder
    u4 = layers.UpSampling2D()(bn)
    u4 = layers.Concatenate()([u4, c4])
    u4 = conv_block(u4, 256)

    u3 = layers.UpSampling2D()(u4)
    u3 = layers.Concatenate()([u3, c3])
    u3 = conv_block(u3, 128)

    u2 = layers.UpSampling2D()(u3)
    u2 = layers.Concatenate()([u2, c2])
    u2 = conv_block(u2, 64)

    u1 = layers.UpSampling2D()(u2)
    u1 = layers.Concatenate()([u1, c1])
    u1 = conv_block(u1, 32)

    outputs = layers.Conv2D(output_channels, 1, activation="softmax")(u1)

    return Model(inputs, outputs)



Note that the number of filters on the last layer is set to the number of `output_channels`. This will be one output channel per class.

## Train the model

Now, all that is left to do is to compile and train the model.

Since this is a multiclass classification problem, use the `tf.keras.losses.SparseCategoricalCrossentropy` loss function with the `from_logits` argument set to `True`, since the labels are scalar integers instead of vectors of scores for each pixel of every class.

When running inference, the label assigned to the pixel is the channel with the highest value. This is what the `create_mask` function is doing.

In [ ]:
import os

print(os.listdir("/content"))



In [ ]:
!find /content -name "*.keras"
!find /content -name "*.h5"



In [ ]:
model = tf.keras.models.load_model(
    "/content/resistor_unet_final (1).keras",
    compile=False
)



In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=total_loss,
    metrics=["accuracy"]
)





Plot the resulting model architecture:

In [ ]:
tf.keras.utils.plot_model(model, show_shapes=True, expand_nested=True, dpi=64)

Try out the model to check what it predicts before training:

In [ ]:
def create_mask(pred_mask):
  pred_mask = tf.math.argmax(pred_mask, axis=-1)
  pred_mask = pred_mask[..., tf.newaxis]
  return pred_mask[0]

In [ ]:
def apply_color_from_ids(mask_ids):
    """
    Convert class ID mask → RGB mask using your resistor real colors.
    """
    h, w = mask_ids.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)

    for cid in np.unique(mask_ids):
        name = ID_TO_NAME.get(int(cid), "Background")
        rgb = RESISTOR_COLORS_RGB[name]
        out[mask_ids == cid] = rgb

    return out


def show_predictions(dataset, num_batches=1):
    """
    Displays predictions for batches from train/val dataset.
    """
    for imgs, masks in dataset.take(num_batches):

        preds = model.predict(imgs)

        for i in range(imgs.shape[0]):
            img = (imgs[i].numpy() * 255).astype(np.uint8)

            # True mask
            true_ids = masks[i].numpy().squeeze()
            true_color = apply_color_from_ids(true_ids)

            # Predicted mask
            pred_ids = np.argmax(preds[i], axis=-1).astype(np.int32)
            pred_color = apply_color_from_ids(pred_ids)

            plt.figure(figsize=(16,5))

            plt.subplot(1,4,1)
            plt.imshow(img)
            plt.title("Original Image")
            plt.axis("off")

            plt.subplot(1,4,2)
            plt.imshow(true_color)
            plt.title("Ground Truth Mask")
            plt.axis("off")

            plt.subplot(1,4,3)
            plt.imshow(pred_color)
            plt.title("Predicted Mask")
            plt.axis("off")

            plt.subplot(1,4,4)
            plt.imshow(img)
            plt.imshow(pred_color, alpha=0.5)
            plt.title("Overlay (Prediction)")
            plt.axis("off")

            plt.show()


In [ ]:
show_predictions(val_ds, num_batches=1)


The callback defined below is used to observe how the model improves while it is training:

In [ ]:
class DisplayCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
    clear_output(wait=True)
    show_predictions()
    print ('\nSample Prediction after epoch {}\n'.format(epoch+1))

In [ ]:
for imgs, masks in train_ds.take(1):
    print("Mask dtype:", masks.dtype)
    print("Unique mask IDs:", np.unique(masks.numpy()))


In [ ]:
class ShowPredictionCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"\n--- Predictions after epoch {epoch+1} ---")
        for imgs, masks in val_ds.take(1):
            preds = model.predict(imgs)
            pred_ids = tf.argmax(preds, axis=-1)[0].numpy()
            true_ids = masks[0].numpy().squeeze()

            print("Unique TRUE:", np.unique(true_ids))
            print("Unique PRED:", np.unique(pred_ids))
            break


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,

)




In [ ]:
import scipy.ndimage as nd
import numpy as np

def clean_pred_mask(mask_ids):
    """
    mask_ids = (H,W) predicted integer mask
    """
    cleaned = np.zeros_like(mask_ids)

    for cid in np.unique(mask_ids):
        if cid == 0:
            continue  # keep background separate

        # extract this class region
        cls_mask = (mask_ids == cid)

        # morphological open + close
        cls_mask = nd.binary_opening(cls_mask, structure=np.ones((3,3)))
        cls_mask = nd.binary_closing(cls_mask, structure=np.ones((5,5)))

        # remove tiny noisy blobs
        cls_mask = nd.binary_fill_holes(cls_mask)
        cls_mask = nd.binary_erosion(cls_mask, iterations=1)

        # keep only if region large enough (remove tiny predictions)
        if cls_mask.sum() > 50:
            cleaned[cls_mask] = cid

    return cleaned



In [ ]:
!pip install scipy


In [ ]:
# --- Final Evaluation (run AFTER training finishes) ---

def apply_color_from_ids(mask_ids):
    h, w = mask_ids.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    for cid in np.unique(mask_ids):
        name = ID_TO_NAME.get(int(cid), "Background")
        rgb = RESISTOR_COLORS_RGB.get(name, [0,0,0])
        out[mask_ids == cid] = rgb
    return out

def show_final_predictions(dataset, num_images=8):
    import numpy as np
    import matplotlib.pyplot as plt

    # Take 1 batch
    for imgs, masks in dataset.take(1):
        preds = model.predict(imgs)

        count = min(num_images, imgs.shape[0])

        for i in range(count):
            img = (imgs[i].numpy() * 255).astype(np.uint8)
            true_ids = np.squeeze(masks[i].numpy())
            pred_ids = np.argmax(preds[i], axis=-1).astype(np.int32)
# ------------ TEMP FIX FOR ORANGE/RED/VIOLET CONFUSION ------------
            pred_ids_original = pred_ids.copy()

# Apply your cleanup function
            pred_ids = clean_pred_mask(pred_ids)

# Restore small correct segments
            pred_ids[pred_ids_original == 2] = 2   # orange
            pred_ids[pred_ids_original == 11] = 11 # red
            pred_ids[pred_ids_original == 12] = 12 # violet
# -----------------------------------------------------------------
            pred_color = apply_color_from_ids(pred_ids)


            true_rgb = apply_color_from_ids(true_ids)
            pred_rgb = apply_color_from_ids(pred_ids)

            print(f"IMAGE {i+1}")
            print("Unique TRUE:", np.unique(true_ids))
            print("Unique PRED:", np.unique(pred_ids))

            plt.figure(figsize=(18,6))
            plt.subplot(1,3,1)
            plt.imshow(img)
            plt.title("Image")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(true_rgb)
            plt.title("True Mask")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(pred_rgb)
            plt.title("Predicted Mask")
            plt.axis("off")

            plt.show()

# Call this AFTER training ends:
# show_final_predictions(val_ds, num_images=3)


In [ ]:
def snap_colors(mask_ids):
    corrected = mask_ids.copy()

    # ---------------------------------------------------------
    # 1. Remove BLUE from ORANGE regions
    # ---------------------------------------------------------
    blue = (mask_ids == 6)
    orange = (mask_ids == 2)

    # If blue exists inside orange → snap it to orange
    corrected[blue & orange] = 2

    # Also remove *nearby* blue speckles by snapping small isolated blue regions
    # (blue pixels with no blue neighbors become orange)
    from scipy.ndimage import binary_erosion
    blue_isolated = blue & ~binary_erosion(blue, structure=np.ones((3,3)))
    corrected[blue_isolated & orange] = 2

    # ---------------------------------------------------------
    # 2. Gold (1) vs Grey (10)
    # ---------------------------------------------------------
    gold = (mask_ids == 1)
    grey = (mask_ids == 10)

    # If grey overlaps gold → turn to gold
    corrected[grey & gold] = 1

    # If grey near gold → also gold
    grey_near_gold = grey & binary_erosion(gold, structure=np.ones((5,5)))
    corrected[grey_near_gold] = 1

    # ---------------------------------------------------------
    # 3. Purple (12) incorrectly placed → convert to blue (6)
    # ---------------------------------------------------------
    corrected[mask_ids == 12] = 6

    # ---------------------------------------------------------
    # 4. White (9) should be grey (10)
    # ---------------------------------------------------------
    corrected[mask_ids == 9] = 10

    # ---------------------------------------------------------
    # 5. Ensure orange remains orange, red remains red
    # ---------------------------------------------------------
    corrected[mask_ids == 2] = 2
    corrected[mask_ids == 11] = 11

    return corrected



In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8,5))
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
show_final_predictions(val_ds, num_images=8)


In [ ]:
for img, mask in train_ds.take(1):
    print("mask dtype:", mask.dtype)
    print("unique:", np.unique(mask.numpy()))


In [ ]:
model.save("resistor_unet_good.keras")



In [ ]:
model.save_weights("resistor_unet_good.weights.h5")


In [ ]:
model.save("/content/resistor_unet_final.keras")


In [ ]:
model.save_weights("/content/resistor_unet_good.weights.h5")


In [ ]:
model.save("resistor_unet_final.keras")
model.save_weights("resistor_unet_final.weights.h5")
print("Model saved.")


In [ ]:
from google.colab import files

files.download("resistor_unet_final.keras")
files.download("resistor_unet_final.weights.h5")


## Make predictions

Now, make some predictions. In the interest of saving time, the number of epochs was kept small, but you may set this higher to achieve more accurate results.

In [ ]:
show_predictions(val_ds, num_batches=6)


In [ ]:
from google.colab import files
files.download("resistor_unet_good.keras")
files.download("resistor_unet_good.weights.h5")


In [ ]:
import os

print(os.listdir())


## Optional: Imbalanced classes and class weights

Semantic segmentation datasets can be highly imbalanced meaning that particular class pixels can be present more inside images than that of other classes. Since segmentation problems can be treated as per-pixel classification problems, you can deal with the imbalance problem by weighing the loss function to account for this. It's a simple and elegant way to deal with this problem. Refer to the [Classification on imbalanced data](../structured_data/imbalanced_data.ipynb) tutorial to learn more.

To [avoid ambiguity](https://github.com/keras-team/keras/issues/3653#issuecomment-243939748), `Model.fit` does not support the `class_weight` argument for targets with 3+ dimensions.

In [ ]:
try:
  model_history = model.fit(train_batches, epochs=EPOCHS,
                            steps_per_epoch=STEPS_PER_EPOCH,
                            class_weight = {0:2.0, 1:2.0, 2:1.0})
  assert False
except Exception as e:
  print(f"Expected {type(e).__name__}: {e}")

So, in this case you need to implement the weighting yourself. You'll do this using sample weights: In addition to `(data, label)` pairs, `Model.fit` also accepts `(data, label, sample_weight)` triples.

Keras `Model.fit` propagates the `sample_weight` to the losses and metrics, which also accept a `sample_weight` argument. The sample weight is multiplied by the sample's value before the reduction step. For example:

In [ ]:
label = np.array([0,0])
prediction = np.array([[-3., 0], [-3, 0]])
sample_weight = [1, 10]

loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)
loss(label, prediction, sample_weight).numpy()

So, to make sample weights for this tutorial, you need a function that takes a `(data, label)` pair and returns a `(data, label, sample_weight)` triple where the `sample_weight` is a 1-channel image containing the class weight for each pixel.

The simplest possible implementation is to use the label as an index into a `class_weight` list:

In [ ]:
def add_sample_weights(image, label):
  # The weights for each class, with the constraint that:
  #     sum(class_weights) == 1.0
  class_weights = tf.constant([2.0, 2.0, 1.0])
  class_weights = class_weights/tf.reduce_sum(class_weights)

  # Create an image of `sample_weights` by using the label at each pixel as an
  # index into the `class weights` .
  sample_weights = tf.gather(class_weights, indices=tf.cast(label, tf.int32))

  return image, label, sample_weights

The resulting dataset elements contain 3 images each:

Now, you can train a model on this weighted dataset:

In [ ]:
weighted_model = unet_model(OUTPUT_CLASSES)
weighted_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'])

## Next steps

Now that you have an understanding of what image segmentation is and how it works, you can try this tutorial out with different intermediate layer outputs, or even different pretrained models. You may also challenge yourself by trying out the [Carvana](https://www.kaggle.com/c/carvana-image-masking-challenge/overview) image masking challenge hosted on Kaggle.

You may also want to see the [Tensorflow Object Detection API](https://github.com/tensorflow/models/blob/master/research/object_detection/README.md) for another model you can retrain on your own data. Pretrained models are available on [TensorFlow Hub](https://www.tensorflow.org/hub/tutorials/tf2_object_detection#optional).